# Finding and Removing Duplicates

Duplicate rows often appear in datasets when scraping websites, combining multiple reports, or because of duplicate submissions. Leaving duplicates in your dataset will skew your statistics (such as inflating your row count, sums, and averages).

Pandas provides two very simple and optimized methods to deal with duplicate values:
* **`.duplicated()`**: Identifies which rows are duplicates by returning a True/False mask.
* **`.drop_duplicates()`**: Instantly removes duplicate rows from your DataFrame.

### Plain English Explanation & Real-Life Analogy
Imagine you are compiling a **VIP Guest List** for a software conference.
* Spongebob and Patrick signed up multiple times.
* When you look at your list, you want to identify who registered more than once.
* By default, you decide to **keep the first registration** (`keep='first'`) and delete any registration that comes after.
* Or, you might decide to **keep the last registration** (`keep='last'`) because it contains the newest information.
* If you are strictly security-focused, you might want to **delete everyone who duplicated** entirely (`keep=False`) to manually review them.

### Code Examples

Let's create a DataFrame with duplicate employee entries to explore these concepts.


In [1]:
import pandas as pd

data = {
    'Name': ['Spongebob', 'Patrick', 'Spongebob', 'Squidward', 'Spongebob'],
    'Age': [30, 35, 30, 50, 30],
    'Job': ['Cook', 'Cashier', 'Cook', 'Cashier', 'Cook']
}

df = pd.DataFrame(data)
print(df)

        Name  Age      Job
0  Spongebob   30     Cook
1    Patrick   35  Cashier
2  Spongebob   30     Cook
3  Squidward   50  Cashier
4  Spongebob   30     Cook


#### Identifying Duplicates with `.duplicated()`
By default, `.duplicated()` marks the **first occurrence** of a row as `False` (not a duplicate) and marks any **subsequent identical rows** as `True` (duplicates).


In [2]:
# Show which rows are considered duplicates
print("--- Duplicated Rows Mask ---")
print(df.duplicated())

# Filter and view the duplicate rows
print("--- View Duplicated Rows ---")
print(df[df.duplicated()])

--- Duplicated Rows Mask ---
0    False
1    False
2     True
3    False
4     True
dtype: bool
--- View Duplicated Rows ---
        Name  Age   Job
2  Spongebob   30  Cook
4  Spongebob   30  Cook


#### Removing Duplicates with `.drop_duplicates()`
To remove duplicates cleanly, use `.drop_duplicates()`.


In [3]:
# Drop duplicates, keeping the first occurrence (default)
df_unique = df.drop_duplicates()
print("--- Drop Duplicates (Default) ---")
print(df_unique)

--- Drop Duplicates (Default) ---
        Name  Age      Job
0  Spongebob   30     Cook
1    Patrick   35  Cashier
3  Squidward   50  Cashier


#### C) Customizing with the `keep` and `subset` Parameters
* **`keep='last'`**: Deletes older entries and keeps the last occurrence.
* **`keep=False`**: Drops **all** occurrences of duplicate rows.
* **`subset`**: Deletes rows based on duplicate values in **specific columns** rather than the entire row.


In [5]:
# Example 1: Keep only the LAST duplicate
df_keep_last = df.drop_duplicates(keep='last')
print("--- Keep Last Duplicate ---")
print(df_keep_last)

# Example 2: Drop based on a subset (e.g., only keep unique Jobs)
df_unique_jobs = df.drop_duplicates(subset=['Job'])
print("--- Unique Jobs (Subset) ---")
print(df_unique_jobs)

--- Keep Last Duplicate ---
        Name  Age      Job
1    Patrick   35  Cashier
3  Squidward   50  Cashier
4  Spongebob   30     Cook
--- Unique Jobs (Subset) ---
        Name  Age      Job
0  Spongebob   30     Cook
1    Patrick   35  Cashier


### Common Pitfalls

1. **Assuming All Columns Must Match**: If one cell in a row is slightly different (e.g., a spelling typo), Pandas will **not** count that row as a duplicate. If you want to find duplicates based on core identifiers (like `Name` or `ID`), you **must** use the `subset` parameter!
2. **Messy Indexes After Dropping**: When you drop duplicates, the old index numbers remain. For example, after dropping Spongebob's duplicates, the index jumps from `1` to `3`. Always use `.reset_index(drop=True)` to rebuild a clean sequential index.


In [6]:
df_cleaned_indexed = df.drop_duplicates().reset_index(drop=True) 
print(df_cleaned_indexed)

        Name  Age      Job
0  Spongebob   30     Cook
1    Patrick   35  Cashier
2  Squidward   50  Cashier


#### Exercise 1 (Easy)
You have a Series of city names: `cities = pd.Series(['Yerevan', 'London', 'Yerevan', 'New York', 'London'])`.
Drop the duplicate cities so each city is listed only once.

In [8]:
import pandas as pd
cities = pd.Series(['Yerevan', 'London', 'Yerevan', 'New York', 'London'])

cities_unique = cities.drop_duplicates() 
print(cities_unique)

0     Yerevan
1      London
3    New York
dtype: str



#### Exercise 2 (Medium)
You have a DataFrame of transactions with timestamp information:
```python
transactions = pd.DataFrame({
    'CustomerID': [101, 102, 101, 103, 101],
    'Amount': [50, 150, 55, 200, 50],
    'Status': ['Pending', 'Completed', 'Completed', 'Completed', 'Completed']
})
```
Write Pandas code to:
1. Drop completely identical transaction rows.
2. Remove duplicate transactions for each `CustomerID`, keeping only the **last** transaction.


In [10]:
import pandas as pd

transactions = pd.DataFrame({
    'CustomerID': [101, 102, 101, 103, 101],
    'Amount': [50, 150, 55, 200, 50],
    'Status': ['Pending', 'Completed', 'Completed', 'Completed', 'Completed']
})

# 1. Drop completely identical rows (Index 4 is a duplicate of Index 0 except Status)
# Wait, let's look at index 0 and index 4: Both have CustomerID 101, Amount 50. 
# But index 0 is 'Pending' and index 4 is 'Completed'. They are NOT completely identical.
# If we drop identical rows:
df_dedup = transactions.drop_duplicates()
print("--- Dropped Completely Identical Rows ---")
print(df_dedup)

# 2. Keep the last transaction for each CustomerID
df_last_tx = transactions.drop_duplicates(subset=['CustomerID'], keep='last').reset_index(drop=True)
print("--- Last Transaction Per Customer ---")
print(df_last_tx)

--- Dropped Completely Identical Rows ---
   CustomerID  Amount     Status
0         101      50    Pending
1         102     150  Completed
2         101      55  Completed
3         103     200  Completed
4         101      50  Completed
--- Last Transaction Per Customer ---
   CustomerID  Amount     Status
0         102     150  Completed
1         103     200  Completed
2         101      50  Completed
